In [ ]:
!pip install python-telegram-bot==20.7 nest_asyncio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.6/552.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.25.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.25.2 which is incompatible.
mcp 1.27.0 requires httpx>=0.27.1, but you have httpx 0.25.2 which is incompatible.
google-adk 1.29.0 requires httpx<1.0.0,>=0.27.0, but you have httpx 0.25.2 which is incompatible.


In [ ]:
%%writefile survey.py
from telegram import InlineKeyboardButton, InlineKeyboardMarkup

QUESTIONS = [
    {"key": "skin_type", "text": "💧 Step 1/11: What is your skin type?", "options": ["Dry", "Oily", "Combination", "Normal", "Sensitive"]},
    {"key": "problems", "text": "🔍 Step 2/11: Main skin concern?", "options": ["Acne/Post-acne", "Wrinkles", "Pigmentation", "Large pores", "Dryness", "Dull complexion"]},
    {"key": "age", "text": "🎂 Step 3/11: Your age?", "options": ["Under 20", "20-30", "30-40", "40+"]},
    {"key": "budget", "text": "💰 Step 4/11: Your budget?", "options": ["Budget", "Mid-range", "Premium"]},
    {"key": "allergies", "text": "⚠️ Step 5/11: Any ingredient allergies?", "options": ["None", "Retinol", "Acids", "Fragrances", "Not sure"]},
    {"key": "climate", "text": "🌍 Step 6/11: Your climate?", "options": ["Dry", "Humid", "Moderate", "Cold"]},
    {"key": "current_products", "text": "🧴 Step 7/11: What do you already use?", "options": ["Nothing", "Cleanser", "Cleanser+moisturizer", "Cleanser+toner+cream", "Cleanser+toner+serum", "Enzyme+toner+serum", "Full routine"]},
    {"key": "sun_exposure", "text": "☀️ Step 8/11: How often are you in the sun?", "options": ["Rarely (indoors)", "Sometimes", "Often outside", "Very often"]},
    {"key": "sleep_stress", "text": "💤 Step 9/11: Sleep & stress?", "options": ["Sleep well, low stress", "Occasionally tired", "Chronic sleep deprivation", "High stress"]},
    {"key": "vitamins", "text": "💊 Step 10/11: Do you take vitamins? (pick multiple, then ✅ Done)", "options": ["None", "Vitamin C", "Vitamin D", "Vitamin E", "Omega-3", "Collagen", "Biotin", "Zinc", "Iron", "Magnesium", "✅ Done"]},
    {"key": "wash_frequency", "text": "🚿 Step 11/11: How often do you cleanse?", "options": ["Once a day", "Morning & evening", "Evening only", "Rarely"]}
]

MULTI_SELECT_QUESTIONS = {"vitamins"}

def get_keyboard(i, selected=None):
    options = QUESTIONS[i]["options"]
    keyboard = []
    for opt in options:
        if opt == "✅ Done":
            keyboard.append([InlineKeyboardButton("✅ Done", callback_data=f"ans_{i}_DONE")])
        else:
            mark = "☑️ " if selected and opt in selected else ""
            keyboard.append([InlineKeyboardButton(f"{mark}{opt}", callback_data=f"ans_{i}_{opt}")])
    return InlineKeyboardMarkup(keyboard)

def get_question_text(i):
    return QUESTIONS[i]["text"]

def is_last_question(i):
    return i >= len(QUESTIONS) - 1

def is_multi_select(i):
    return QUESTIONS[i]["key"] in MULTI_SELECT_QUESTIONS

Writing survey.py


In [ ]:
%%writefile recommender.py
PRODUCTS = {
    "Budget": {
        "cleanser": ("CeraVe Foaming Cleanser", 7500),
        "toner": ("Pyunkang Yul Toner", 7000),
        "serum": ("The Ordinary Niacinamide 10%", 7500),
        "moisturizer": ("Neutrogena Hydro Boost", 8000),
        "spf": ("Garnier Ambre Solaire SPF50", 7500),
        "oil_cleanser": ("Garnier Micellar Water", 7000),
        "exfoliant": ("The Ordinary AHA+BHA", 8000),
        "retinol": ("The Ordinary Retinol 0.5%", 7500)
    },
    "Mid-range": {
        "cleanser": ("La Roche-Posay Toleriane", 15000),
        "toner": ("Paula's Choice Toner", 16000),
        "serum": ("Paula's Choice Niacinamide", 17000),
        "moisturizer": ("Kiehl's Ultra Facial Cream", 19000),
        "spf": ("La Roche-Posay Anthelios", 16000),
        "oil_cleanser": ("Clinique Take The Day Off", 18000),
        "exfoliant": ("Paula's Choice BHA Liquid", 17000),
        "retinol": ("RoC Retinol Correxion", 15000)
    },
    "Premium": {
        "cleanser": ("Tatcha The Rice Wash", 65000),
        "toner": ("SK-II Facial Treatment Essence", 95000),
        "serum": ("Skinceuticals C E Ferulic", 110000),
        "moisturizer": ("La Mer Creme de la Mer", 120000),
        "spf": ("Supergoop! Unseen Sunscreen SPF40", 70000),
        "oil_cleanser": ("Shu Uemura Cleansing Oil", 75000),
        "exfoliant": ("Drunk Elephant T.L.C. Framboos", 85000),
        "retinol": ("Drunk Elephant A-Passioni Retinol", 90000)
    }
}

def build_routine(answers):
    budget = answers.get("budget", "Budget")
    p = PRODUCTS.get(budget, PRODUCTS["Budget"])
    allergy = answers.get("allergies", "None")

    morning_items = [
        ("Cleanser", p['cleanser']),
        ("Toner", p['toner']),
        ("Serum", p['serum']),
        ("Moisturizer", p['moisturizer']),
        ("SPF", p['spf'])
    ]

    retinol = p['retinol'] if allergy != "Retinol" else ("Niacinamide Serum (retinol excluded)", 0)
    evening_items = [
        ("Makeup remover", p['oil_cleanser']),
        ("Cleanser", p['cleanser']),
        ("Exfoliant (2-3x/week)", p['exfoliant']),
        ("Actives", retinol),
        ("Moisturizer", p['moisturizer'])
    ]

    morning_text = ""
    for i, (step, (name, price)) in enumerate(morning_items, 1):
        morning_text += f"{i}️⃣ {step}: {name} — {price:,} KZT\n"

    evening_text = ""
    for i, (step, (name, price)) in enumerate(evening_items, 1):
        evening_text += f"{i}️⃣ {step}: {name} — {price:,} KZT\n"

    seen = set()
    unique_items = []
    total = 0
    for _, (name, price) in morning_items + evening_items:
        if name not in seen:
            seen.add(name)
            unique_items.append((name, price))
            total += price

    products_list = "\n".join([f"• {name} — {price:,} KZT" for name, price in unique_items])

    routine = (
        f"☀️ MORNING ROUTINE:\n{morning_text}\n"
        f"🌙 EVENING ROUTINE:\n{evening_text}\n"
        f"━━━━━━━━━━━━━━━\n"
        f"🛒 All products & prices:\n{products_list}\n\n"
        f"💰 Total: ~{total:,} KZT"
    )
    return routine

Writing recommender.py


In [ ]:
%%writefile database.py
import json, os

def save_routine(user_id, answers, routine):
    db = {}
    if os.path.exists("routines.json"):
        with open("routines.json") as f:
            db = json.load(f)
    db[str(user_id)] = {"answers": answers, "routine": routine}
    with open("routines.json", "w") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)

def get_routine(user_id):
    if not os.path.exists("routines.json"):
        return None
    with open("routines.json") as f:
        db = json.load(f)
    return db.get(str(user_id))

def delete_routine(user_id):
    if not os.path.exists("routines.json"):
        return False
    with open("routines.json") as f:
        db = json.load(f)
    if str(user_id) in db:
        del db[str(user_id)]
        with open("routines.json", "w") as f:
            json.dump(db, f, ensure_ascii=False, indent=2)
        return True
    return False

Writing database.py


In [ ]:
%%writefile main.py
import nest_asyncio
nest_asyncio.apply()
import json, os, logging
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import ApplicationBuilder, CommandHandler, CallbackQueryHandler, ContextTypes
from survey import get_keyboard, get_question_text, is_last_question, is_multi_select, QUESTIONS
from recommender import build_routine
from database import save_routine, get_routine

BOT_TOKEN = "8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk"
WHATSAPP_NUMBER = "77055130486"

logging.basicConfig(level=logging.INFO)

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    context.user_data.clear()
    context.user_data["answers"] = {}
    context.user_data["q"] = 0
    context.user_data["multi_selected"] = []
    await update.message.reply_text(
    "👋 Hi! I will create a personalized skincare routine just for you. Let's go! 🚀\n\n"
    "📌 Available commands:\n"
    "/start — Take the survey\n"
    "/myroutine — View your saved routine\n"
    "/ingredient [name] — Learn about an ingredient\n"
    "   Example: /ingredient niacinamide\n"
    "/sos — Quick help for skin emergencies 🆘"
)
    await update.message.reply_text(get_question_text(0), reply_markup=get_keyboard(0))

async def my_routine(update: Update, context: ContextTypes.DEFAULT_TYPE):
    saved = get_routine(update.effective_user.id)
    if saved:
        await update.message.reply_text("📋 Your saved routine:\n\n" + saved["routine"])
    else:
        await update.message.reply_text("No routine saved yet. Take the survey: /start")

async def handle_answer(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()

    if query.data == "buy_yes":
        url = f"https://wa.me/{WHATSAPP_NUMBER}?text=Hi!+I+want+to+order+my+skincare+routine"
        await query.message.reply_text(f"🛍 Amazing! Contact us on WhatsApp:\n{url}")
        return
    elif query.data == "buy_maybe":
        await query.message.reply_text("🤔 No worries! Your routine is saved. Come back anytime — /myroutine 💾")
        return
    elif query.data == "buy_no":
        await query.message.reply_text("👌 Routine saved. Come back whenever you're ready! 😊")
        return

    try:
        _, q_str, answer = query.data.split("_", 2)
        q = int(q_str)
    except Exception:
        return

    if q != context.user_data.get("q", 0):
        return

    if is_multi_select(q):
        if answer == "DONE":
            selected = context.user_data.get("multi_selected", [])
            if not selected:
                selected = ["None"]
            context.user_data["answers"][QUESTIONS[q]["key"]] = ", ".join(selected)
            await query.edit_message_text(f"✅ {get_question_text(q)}\n→ {', '.join(selected)}")
            context.user_data["multi_selected"] = []
            next_q = q + 1
            context.user_data["q"] = next_q
            if is_last_question(q):
                await finish(query, context)
            else:
                await query.message.reply_text(get_question_text(next_q), reply_markup=get_keyboard(next_q))
        else:
            selected = context.user_data.get("multi_selected", [])
            if answer in selected:
                selected.remove(answer)
            else:
                selected.append(answer)
            context.user_data["multi_selected"] = selected
            try:
                await query.edit_message_reply_markup(reply_markup=get_keyboard(q, selected))
            except Exception:
                pass
        return

    context.user_data["answers"][QUESTIONS[q]["key"]] = answer
    await query.edit_message_text(f"✅ {get_question_text(q)}\n→ {answer}")
    next_q = q + 1
    context.user_data["q"] = next_q

    if is_last_question(q):
        await finish(query, context)
    else:
        await query.message.reply_text(get_question_text(next_q), reply_markup=get_keyboard(next_q))

async def finish(query, context):
    routine = build_routine(context.user_data["answers"])
    save_routine(query.from_user.id, context.user_data["answers"], routine)
    await query.message.reply_text("🎉 Your personalized skincare routine:\n\n" + routine)
    keyboard = InlineKeyboardMarkup([
        [InlineKeyboardButton("✅ Ready to buy!", callback_data="buy_yes")],
        [InlineKeyboardButton("🤔 I'll think about it", callback_data="buy_maybe")],
        [InlineKeyboardButton("❌ Not right now", callback_data="buy_no")]
    ])
    await query.message.reply_text("🛒 Would you like to order these products?", reply_markup=keyboard)

async def error_handler(update, context):
    logging.error(f"Error: {context.error}")

INGREDIENTS = {
    "niacinamide": "💊 Niacinamide (Vitamin B3)\n✅ Minimizes pores, brightens skin, reduces redness\n⚠️ Don't mix with: Vitamin C (use at different times)\n👍 Good for: oily, combination, acne-prone skin",
    "retinol": "💊 Retinol (Vitamin A)\n✅ Reduces wrinkles, boosts collagen, fights acne\n⚠️ Don't mix with: AHA/BHA, Vitamin C\n👍 Good for: aging, acne, pigmentation\n🌙 Use only at night!",
    "vitamin c": "💊 Vitamin C (Ascorbic Acid)\n✅ Brightens, fades dark spots, antioxidant\n⚠️ Don't mix with: Retinol, Niacinamide\n👍 Good for: dull skin, pigmentation\n☀️ Use in the morning!",
    "hyaluronic acid": "💊 Hyaluronic Acid\n✅ Deep hydration, plumps skin, holds moisture\n⚠️ Apply on damp skin for best results\n👍 Good for: all skin types especially dry",
    "salicylic acid": "💊 Salicylic Acid (BHA)\n✅ Unclogs pores, fights acne, exfoliates\n⚠️ Don't mix with: Retinol\n👍 Good for: oily, acne-prone, large pores\n🌙 Use 2-3x per week",
    "glycolic acid": "💊 Glycolic Acid (AHA)\n✅ Exfoliates dead skin, brightens, smooths texture\n⚠️ Don't mix with: Retinol, Vitamin C\n👍 Good for: dull skin, uneven texture\n🌙 Use 2-3x per week at night",
    "ceramides": "💊 Ceramides\n✅ Restores skin barrier, locks in moisture\n⚠️ Safe to mix with everything!\n👍 Good for: dry, sensitive, damaged skin",
    "spf": "💊 SPF (Sunscreen)\n✅ Protects from UV damage, prevents aging & pigmentation\n⚠️ Must reapply every 2 hours outdoors\n👍 Good for: everyone, every single day! ☀️",
    "peptides": "💊 Peptides\n✅ Boosts collagen, firms skin, reduces wrinkles\n⚠️ Safe to mix with most ingredients\n👍 Good for: aging skin, loss of elasticity",
    "azelaic acid": "💊 Azelaic Acid\n✅ Fades dark spots, fights acne, reduces redness\n⚠️ Safe to mix with most ingredients\n👍 Good for: rosacea, pigmentation, acne",
    "squalane": "💊 Squalane\n✅ Deep moisturizing, anti-aging, non-comedogenic\n⚠️ Safe to mix with everything!\n👍 Good for: all skin types especially dry & sensitive",
    "zinc": "💊 Zinc\n✅ Controls oil, fights acne, soothes inflammation\n⚠️ Safe to mix with most ingredients\n👍 Good for: oily, acne-prone skin"
}

async def ingredient(update: Update, context: ContextTypes.DEFAULT_TYPE):
    keyboard = InlineKeyboardMarkup([
        [InlineKeyboardButton("Niacinamide", callback_data="ing_niacinamide"),
         InlineKeyboardButton("Retinol", callback_data="ing_retinol")],
        [InlineKeyboardButton("Vitamin C", callback_data="ing_vitamin c"),
         InlineKeyboardButton("Hyaluronic Acid", callback_data="ing_hyaluronic acid")],
        [InlineKeyboardButton("Salicylic Acid", callback_data="ing_salicylic acid"),
         InlineKeyboardButton("Glycolic Acid", callback_data="ing_glycolic acid")],
        [InlineKeyboardButton("Ceramides", callback_data="ing_ceramides"),
         InlineKeyboardButton("Peptides", callback_data="ing_peptides")],
        [InlineKeyboardButton("Azelaic Acid", callback_data="ing_azelaic acid"),
         InlineKeyboardButton("Squalane", callback_data="ing_squalane")],
        [InlineKeyboardButton("SPF", callback_data="ing_spf"),
         InlineKeyboardButton("Zinc", callback_data="ing_zinc")]
    ])
    await update.message.reply_text("🔍 Choose an ingredient to learn about:", reply_markup=keyboard)

async def handle_ingredient(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    name = query.data.replace("ing_", "")
    info = INGREDIENTS.get(name)
    if info:
        await query.message.reply_text(info)
    else:
        await query.message.reply_text("❌ Not found.")

async def sos(update: Update, context: ContextTypes.DEFAULT_TYPE):
    keyboard = InlineKeyboardMarkup([
        [InlineKeyboardButton("🔴 Pimple before an event", callback_data="sos_pimple")],
        [InlineKeyboardButton("🔥 Skin is red & irritated", callback_data="sos_redness")],
        [InlineKeyboardButton("🏜️ Skin is very dry & flaky", callback_data="sos_dry")],
        [InlineKeyboardButton("💧 Oily & shiny skin", callback_data="sos_oily")],
        [InlineKeyboardButton("😖 Breakout all over face", callback_data="sos_breakout")],
        [InlineKeyboardButton("🌞 Sunburn", callback_data="sos_sunburn")]
    ])
    await update.message.reply_text("🆘 What's your skin emergency?", reply_markup=keyboard)

SOS_ANSWERS = {
    "sos_pimple": (
        "🔴 Pimple before an event:\n\n"
        "1. DON'T squeeze it!\n"
        "2. Apply ice wrapped in cloth for 2 min\n"
        "3. Dab salicylic acid or tea tree oil\n"
        "4. Use a pimple patch overnight\n"
        "5. In the morning — green color corrector under foundation\n\n"
        "⏱ Reduces size in 4-6 hours!"
    ),
    "sos_redness": (
        "🔥 Red & irritated skin:\n\n"
        "1. Stop all actives (acids, retinol) TODAY\n"
        "2. Apply cold aloe vera gel\n"
        "3. Use only gentle cleanser + thick moisturizer\n"
        "4. Avoid hot water when washing face\n"
        "5. No makeup until skin calms down\n\n"
        "⏱ Should calm in 24-48 hours"
    ),
    "sos_dry": (
        "🏜️ Very dry & flaky skin:\n\n"
        "1. Don't exfoliate — it will make it worse!\n"
        "2. Apply hyaluronic acid on damp skin\n"
        "3. Layer a thick cream on top immediately\n"
        "4. Use face oil as last step (squalane, rosehip)\n"
        "5. Drink more water today 💧\n\n"
        "⏱ Visible improvement in a few hours!"
    ),
    "sos_oily": (
        "💧 Oily & shiny skin:\n\n"
        "1. Don't over-wash — it makes more oil!\n"
        "2. Use blotting papers during the day\n"
        "3. Apply niacinamide serum in the morning\n"
        "4. Switch to gel moisturizer (no heavy creams)\n"
        "5. Use mattifying SPF\n\n"
        "💡 Tip: oily skin is often dehydrated — hydrate more!"
    ),
    "sos_breakout": (
        "😖 Breakout all over face:\n\n"
        "1. Check if you changed any product recently — stop it!\n"
        "2. Clean your phone screen & pillowcase today\n"
        "3. Use gentle cleanser only — no harsh scrubs\n"
        "4. Apply niacinamide serum all over face\n"
        "5. Don't touch your face!\n"
        "6. Drink water, reduce sugar & dairy for 3 days\n\n"
        "⏱ Give it 3-5 days to calm down"
    ),
    "sos_sunburn": (
        "🌞 Sunburn:\n\n"
        "1. Cool the skin — cold (not ice) water or cloth\n"
        "2. Apply pure aloe vera gel generously\n"
        "3. Use fragrance-free gentle moisturizer\n"
        "4. NO acids, retinol or actives for 1 week!\n"
        "5. Stay out of sun until fully healed\n"
        "6. Take ibuprofen if painful\n\n"
        "⚠️ If blistering — see a doctor!"
    )
}

async def handle_sos(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    answer = SOS_ANSWERS.get(query.data)
    if answer:
        await query.message.reply_text(answer)

app = ApplicationBuilder().token(BOT_TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(CommandHandler("myroutine", my_routine))
app.add_handler(CommandHandler("ingredient", ingredient))
app.add_handler(CallbackQueryHandler(handle_ingredient, pattern=r"^ing_"))
app.add_handler(CommandHandler("sos", sos))
app.add_handler(CallbackQueryHandler(handle_sos, pattern=r"^sos_"))
app.add_handler(CallbackQueryHandler(handle_answer, pattern=r"^ans_"))
app.add_handler(CallbackQueryHandler(handle_answer, pattern=r"^buy_"))
app.add_error_handler(error_handler)
app.run_polling()

Writing main.py


In [6]:
!python main.py

INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/getMe "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/deleteWebhook "HTTP/1.1 200 OK"
INFO:telegram.ext.Application:Application started
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/sendMessage "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/sendMessage "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/sendMessage "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot8638382652:AAHhc0qR2__xGFgOhup3BLd4IEqLmh5AXNk/sendMessage "HTTP/1.1 200 OK"
INFO:httpx

In [11]:
readme = """# Skincare Bot by Caring is Sharing

A Telegram bot that asks 11 questions about your skin and generates a personalized skincare routine.

## Features
- 11-step inline survey
- Personalized morning and evening routines
- Products across 3 budget tiers (Budget / Mid-range / Premium)
- Prices in KZT for every product + total cost
- Save and retrieve routine with /myroutine
- Ingredient encyclopedia with /ingredient
- SOS emergency skin help with /sos
- WhatsApp order button at the end of survey

## Project Structure
- main.py — Bot entry point, all handlers
- survey.py — Survey questions and inline keyboards
- recommender.py — Routine building logic and products
- database.py — JSON-based data persistence
- README.md — This file

## How to Run
1. Open Google Colab: colab.research.google.com
2. Install dependencies: pip install python-telegram-bot==20.7 nest_asyncio
3. Add your bot token in main.py
4. Run: python main.py

## Commands
- /start — Take the 11-step skin survey
- /myroutine — View your saved routine
- /ingredient — Learn about skincare ingredients
- /sos — Emergency skin help

## Budget Tiers
- Budget: 7,000 - 9,000 KZT per product
- Mid-range: 15,000 - 20,000 KZT per product
- Premium: 60,000 - 120,000 KZT per product

## Team
Caring is Sharing
| Name | Module |
| Demegen Amina  |
| Maxutova Gulden|
| Rakhman Aidana|
| Talap Akbota |
"""

with open("README.md", "w") as f:
    f.write(readme)

print("README.md created!")

README.md created!


In [12]:
from google.colab import files
files.download('main.py')
files.download('survey.py')
files.download('recommender.py')
files.download('database.py')
files.download('README.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>